# Placement Prediction and Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import pickle
import os

os.makedirs('../reports', exist_ok=True)
os.makedirs('../screenshots', exist_ok=True)

df = pd.read_csv('../dataset/placement_dataset.csv')
df.head()

In [ ]:
sns.set_theme(style="whitegrid")

# Branch-wise placement rate
plt.figure(figsize=(8, 5))
branch_placement = df.groupby('Branch')['Placement_Status'].value_counts(normalize=True).unstack()
branch_placement.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'])
plt.title('Branch-wise Placement Rate')
plt.ylabel('Proportion')
plt.show()

In [ ]:
# Coding Score vs Placement
plt.figure(figsize=(8, 5))
sns.boxplot(x='Placement_Status', y='Coding_Score', data=df, palette='Set2')
plt.title('Coding Score vs Placement')
plt.show()

In [ ]:
print("Preparing data for ML...")
le_branch = LabelEncoder()
df['Branch_Encoded'] = le_branch.fit_transform(df['Branch'])
df['Placement_Encoded'] = df['Placement_Status'].apply(lambda x: 1 if x == 'Placed' else 0)

features = ['Branch_Encoded', 'CGPA', 'Aptitude_Score', 'Communication_Score', 'Coding_Score', 'Internship_Count', 'Project_Count', 'Backlogs']
X = df[features]
y = df['Placement_Encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

with open('model.pkl', 'wb') as f:
    pickle.dump({
        'model': dt_model,
        'scaler': scaler,
        'le_branch': le_branch,
        'features': features
    }, f)
print("model.pkl saved.")